**Representing Categories in TensorFlow**

In [14]:
import tensorflow as tf

# Example categorical data
brands = tf.constant(["Toyota", "BMW", "Ford", "Toyota"])

brands_2 = tf.constant(["Vovo","Toypta","BMW","Ford","Toyota"])
# StringLookup layer maps categories to integer IDs
# Build the vocabulary from the dataset using adapt()
# StringLookup rules:
# - Index 0 is always reserved for OOV (out-of-vocabulary tokens)
# - Categories are sorted first by frequency (descending)
# - If frequencies tie, tie-breaking is done in reverse alphabetical order

# In this dataset:
# Toyota appears 2 times, BMW and Ford appear 1 time each
# So vocabulary = ['[UNK]', 'Toyota', 'Ford', 'BMW']
# Mapping: Toyota 1, Ford 2, BMW 3

# Create StringLookup layer (maps strings to integer IDs)
lookup = tf.keras.layers.StringLookup(output_mode="int")

# Build vocabulary from the dataset
# lookup.adapt(brands)
lookup.adapt(brands_2)
print(lookup.get_vocabulary())

# Convert categories to numeric IDs
# ids = lookup(brands)
ids = lookup(brands_2)
print("Category IDs:", ids.numpy())

# Expected output: [1, 3, 2, 1]
# Mapping: Toyota -> 1, BMW -> 3, Ford -> 2

['[UNK]', np.str_('Vovo'), np.str_('Toypta'), np.str_('Toyota'), np.str_('Ford'), np.str_('BMW')]
Category IDs: [1 2 5 4 3]


**One-Hot Encoding - Scikit**

In [ ]:
import numpy as np
from sklearn.preprocessing import OneHotEncoder

# Example categorical values
fuel_types = np.array(["Petrol", "Diesel", "Electric", "Petrol"])

# Step 1: Reshape for sklearn
fuel_array = fuel_types.reshape(-1, 1)

# Step 2: OneHotEncoder
ohe = OneHotEncoder(sparse_output=False, drop=None)

one_hot = ohe.fit_transform(fuel_array)

# Step 3: Print results
print("One-Hot Encoded (sklearn):\n", one_hot)
print("Feature Names:", ohe.get_feature_names_out(["fuel_type"]))

One-Hot Encoded (sklearn):
 [[0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
Feature Names: ['fuel_type_Diesel' 'fuel_type_Electric' 'fuel_type_Petrol']


**One-Hot Encoding (Pandas)**

In [ ]:
import pandas as pd

# Example categorical values
fuel_types = ["Petrol", "Diesel", "Electric", "Petrol"]

# Create DataFrame
df = pd.DataFrame({"fuel_type": fuel_types})

# Apply one-hot encoding
one_hot = pd.get_dummies(df["fuel_type"], drop_first=False)

print("One-Hot Encoded (pandas):")
print(one_hot)

One-Hot Encoded (pandas):
   Diesel  Electric  Petrol
0   False     False    True
1    True     False   False
2   False      True   False
3   False     False    True


**One-Hot Encoding (TensorFlow)**

In [ ]:
import tensorflow as tf

# Example categorical values
fuel_types = tf.constant(["Petrol", "Diesel", "Electric", "Petrol"])

# Step 1: Convert strings to integer IDs
lookup = tf.keras.layers.StringLookup()
lookup.adapt(fuel_types)

ids = lookup(fuel_types)

# Step 2: One-Hot Encoding
ohe = tf.keras.layers.CategoryEncoding(
    num_tokens=lookup.vocabulary_size(),
    output_mode="one_hot"
)

one_hot = ohe(ids)

print("One-Hot Encoded (TensorFlow):\n", one_hot.numpy())
print("Vocabulary:", lookup.get_vocabulary())

One-Hot Encoded (TensorFlow):
 [[0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]]
Vocabulary: ['[UNK]', np.str_('Petrol'), np.str_('Electric'), np.str_('Diesel')]


**Label Encoding (Scikit Learn)**

In [ ]:
from sklearn.preprocessing import LabelEncoder

fuel_types = ["Petrol", "Diesel", "Electric", "Petrol"]

le = LabelEncoder()
labels = le.fit_transform(fuel_types)

print("Label Encoded:", labels)
print("Classes:", le.classes_)

Label Encoded: [2 0 1 2]
Classes: ['Diesel' 'Electric' 'Petrol']


**Label Encoding (Pandas)**

In [ ]:
import pandas as pd

fuel_types = ["Petrol", "Diesel", "Electric", "Petrol"]

df = pd.DataFrame({"fuel_type": fuel_types})

df["fuel_type_encoded"] = df["fuel_type"].astype("category").cat.codes

print(df)

  fuel_type  fuel_type_encoded
0    Petrol                  2
1    Diesel                  0
2  Electric                  1
3    Petrol                  2


**Label Encoding + Embeddings (TensorFlow)**

In [ ]:
import tensorflow as tf

fuel_types = tf.constant(["Petrol", "Diesel", "Electric", "Petrol"])

# Label encoding
lookup = tf.keras.layers.StringLookup(output_mode="int")
lookup.adapt(fuel_types)

labels = lookup(fuel_types)

print("Label Encoded:", labels.numpy())
print("Vocabulary:", lookup.get_vocabulary())

# Embedding layer
embedding_dim = 3

embed_layer = tf.keras.layers.Embedding(
    input_dim=lookup.vocabulary_size(),
    output_dim=embedding_dim
)

embeddings = embed_layer(labels)

print("Embeddings:\n", embeddings.numpy())

Label Encoded: [1 3 2 1]
Vocabulary: ['[UNK]', np.str_('Petrol'), np.str_('Electric'), np.str_('Diesel')]
Embeddings:
 [[ 0.00888073 -0.04312282 -0.01475399]
 [-0.00985297 -0.03480808 -0.01297515]
 [-0.01121761  0.02697295  0.04818069]
 [ 0.00888073 -0.04312282 -0.01475399]]


**Bag-of-Words (Scikit-learn)**

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

docs = [
    "I love deep learning",
    "Deep learning loves me"
]

vectorizer = CountVectorizer()

bow = vectorizer.fit_transform(docs)

print("Vocabulary:", vectorizer.get_feature_names_out())
print("Bag-of-Words:\n", bow.toarray())

Vocabulary: ['deep' 'learning' 'love' 'loves' 'me']
Bag-of-Words:
 [[1 1 1 0 0]
 [1 1 0 1 1]]


**TF-IDF Example (Scikit-learn)**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

docs = [
    "I love deep learning",
    "Deep learning loves me",
    "Team Taiwan No1.",
    "Taiwan Team"
]

vectorizer = TfidfVectorizer(
    token_pattern=r"(?u)\b\w+\b",
    use_idf=True
)

tfidf = vectorizer.fit_transform(docs)

print("Vocabulary:", vectorizer.get_feature_names_out())
print("TF-IDF:\n", np.round(tfidf.toarray(), 3))

Vocabulary: ['deep' 'i' 'learning' 'love' 'loves' 'me' 'no1' 'taiwan' 'team']
TF-IDF:
 [[0.438 0.555 0.438 0.555 0.    0.    0.    0.    0.   ]
 [0.438 0.    0.438 0.    0.555 0.555 0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.668 0.526 0.526]
 [0.    0.    0.    0.    0.    0.    0.    0.707 0.707]]


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

texts = [
    "The patient was admitted with chest pain and shortness of breath.",
    "History of diabetes and hypertension."
]

inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

cls_embeddings = outputs.last_hidden_state[:, 0, :]

print("CLS Embedding shape:", cls_embeddings.shape)
print("First embedding sample:", cls_embeddings[0][:5])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLS Embedding shape: torch.Size([2, 768])
First embedding sample: tensor([ 0.1112,  0.3104, -0.1773,  0.2884,  0.4194])
